# 08 — Dates, timestamps and string-derived fields

**Student exercise**

Complete the tasks in order. Each task includes an expected result, three hints, and a check. All sample data and setup code are included in this notebook.

## 1. Spark configuration

Start a fresh kernel. Use Spark 3.5.x / Scala 2.12 and Java 17. Set `STUDENT_ID` to a lowercase name containing letters, numbers or underscores. Set `SPARK_MASTER=local[2]` for a local Spark session.

In [ ]:
import os
import socket
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

master_url = os.environ.get("SPARK_MASTER", f"spark://{socket.gethostname()}:7077")
STUDENT_ID = "student01"
assert STUDENT_ID and STUDENT_ID[0].islower() and all(c in "abcdefghijklmnopqrstuvwxyz0123456789_" for c in STUDENT_ID)

spark = (
    SparkSession.builder
    .appName("Exercise-08")
    .master(master_url)
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "2")
    .config("spark.redaction.regex", "(?i)secret|password|token|access[.]?key|credential")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

## 2. Sample data

Run this cell, inspect the records, and use the supplied variables in the tasks.

In [ ]:
events=spark.createDataFrame([(1,'2026-09-01 10:00:00',' paid ',40.0),
 (2,'2026-09-02 11:00:00','PAID',50.0),(3,'2026-10-01 09:00:00','cancelled',20.0)],
 'id INT,event_text STRING,status STRING,amount DOUBLE')
movies=spark.createDataFrame([(1,'Toy Story (1995)'),(2,'Untitled')],'movie_id INT,title STRING')

## 3. Result checks

This small function checks output values and duplicate counts. It is included here so the notebook can run on its own.

In [ ]:
def check_frame(df, columns, expected):
    assert df is not None, "Complete the task before running the check"
    assert set(columns).issubset(df.columns), "Missing output columns"
    def normalize(row):
        return tuple(round(value, 6) if isinstance(value, float) else value for value in row)
    actual = sorted(repr(normalize(row)) for row in df.select(*columns).collect())
    wanted = sorted(repr(normalize(row)) for row in expected)
    assert actual == wanted, f"Expected {wanted}, got {actual}"
    print("PASS:", columns)

## Task 1: Parse timestamps and normalize status

Create dated with event_at TIMESTAMP, event_date DATE, month STRING yyyy-MM and normalized lowercase/trimmed status.

**Expected result:** September IDs1,2 and October ID3.

In [ ]:
dated=None  # TODO: to_timestamp, to_date, date_format, lower/trim

**Hint 1:** The session timezone is UTC for consistent teaching results.

**Hint 2:** The input pattern distinguishes MM (month) from mm (minutes).

**Hint 3:** to_timestamp(event_text, pattern), then derive date and month from that typed column.

### Check

In [ ]:
check_frame(dated,['id','month','status'],[(1,'2026-09','paid'),(2,'2026-09','paid'),(3,'2026-10','cancelled')])
assert dict(dated.dtypes)['event_at']=='timestamp'

## Task 2: Revenue by month using SQL

Produce monthly for paid records only, grouped by month.

**Expected result:** September paid revenue90; no October paid row.

In [ ]:
monthly=None  # TODO: WHERE status and GROUP BY month

**Hint 1:** Cancelled events should not enter paid revenue.

**Hint 2:** Filter normalized status before grouping.

**Hint 3:** SELECT month,SUM(amount) AS revenue FROM dated_events WHERE ... GROUP BY month.

### Check

In [ ]:
check_frame(monthly,['month','revenue'],[('2026-09',90.0)])

## Task 3: Extract an optional release year

Create titled with movie_id and release_year INT. Extract four digits in trailing parentheses; no match becomes null.

**Expected result:** Toy Story1995; Untitled null.

In [ ]:
titled=None  # TODO: regexp_extract and try_cast

**Hint 1:** regexp_extract returns an empty string if the pattern does not match.

**Hint 2:** Anchor the closing parenthesis to the end with $.

**Hint 3:** Extract capture group1, then try_cast the result to INT.

### Check

In [ ]:
check_frame(titled,['movie_id','release_year'],[(1,1995),(2,None)])

## More practice

Calculate days between events; derive weekday; compare month truncation with formatted month strings.

## Common mistakes

A timestamp string is not a typed timestamp. Do not turn a missing year into zero. Timezones affect date extraction.

## Finish

Stop Spark before starting another notebook. For write exercises, run setup again before repeating append tasks. S3 data remains available after stopping Spark.

In [ ]:
spark.stop()